# StyleMatch Literary Source-Heldout Audit

Creates a stricter split where dev/test books are held out by source, then reruns the same baseline models.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/stylematch_v1')
assert (Path.cwd() / 'scripts/make_source_heldout_splits.py').exists(), 'Run from cloned style_matching repo.'
assert (Path.cwd() / 'scripts/literary_baseline.py').exists(), 'Run from cloned style_matching repo.'


In [ ]:
!pip -q install pandas pyarrow scikit-learn


In [ ]:
chunks_with_text = ROOT / 'data/literary/meta/chunks_with_text.parquet'
heldout_split = ROOT / 'data/literary/meta/literary_source_heldout_splits.parquet'
heldout_report = ROOT / 'data/literary/meta/literary_source_heldout_report.json'

!python scripts/make_source_heldout_splits.py --input "{chunks_with_text}" --output "{heldout_split}" --report "{heldout_report}"


In [ ]:
import json
import pandas as pd

report = json.loads(heldout_report.read_text())
print('eligible authors:', report['eligible_authors'])
print('excluded authors:', report['excluded_authors'])
df = pd.read_parquet(heldout_split)
display(df.groupby('split').size())
display(df.groupby(['author_or_speaker', 'split']).size().unstack(fill_value=0).sort_values('test'))


In [ ]:
out_dir = ROOT / 'artifacts/literary_source_heldout_baseline'
!python scripts/literary_baseline.py --input "{heldout_split}" --out-dir "{out_dir}"


In [ ]:
metrics = json.loads((out_dir / 'literary_baseline_metrics.json').read_text())
metrics['models']
